# Verification

Is Model A's improvement over the baseline statistically significant and robust? And does Model B (+ reviews) add anything beyond Model A?

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from pathlib import Path

RESULTS_DIR = Path('results')
DATA_DIR    = Path('data')

TS_COLS  = ['ts_mean', 'ts_slope', 'ts_curvature', 'ts_range', 'ts_peak_pos']
REV_COLS = ['rev_count', 'rev_mean_rating', 'rev_positive_frac']
TARGET   = 'actual_peak_value'
ALPHAS   = [0.01, 0.1, 1.0, 10.0, 100.0]
N_PERM   = 1000
N_BOOT   = 500

def mae(actual, predicted):
    return float(np.mean(np.abs(np.array(actual, dtype=float) - np.array(predicted, dtype=float))))

def make_pipe(num_cols):
    ct = ColumnTransformer([
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ['style']),
        ('sc',  StandardScaler(), num_cols),
    ])
    return Pipeline([('ct', ct), ('ridge', RidgeCV(alphas=ALPHAS))])

rng = np.random.default_rng(42)

## Verification strategy

Three questions:

1. Is the Model A vs Baseline difference significant? - Permutation test
2. Is it stable? - Bootstrap CI on the improvement
3. How much early data is needed? - Sensitivity by window length

All tests are non-parametric - they make no assumptions about normality. The permutation test is the gold standard for small samples.

In [10]:
predictions  = pd.read_csv(RESULTS_DIR / 'peak_predictions.csv')
actual    = predictions['actual'].values.astype(float)
pred_a    = predictions['pred_a'].values.astype(float)
pred_base = predictions['pred_base'].values.astype(float)

mae_a    = mae(actual, pred_a)
mae_base = mae(actual, pred_base)
diff     = mae_base - mae_a

print(f'Baseline MAE : {mae_base:.2f} pts')
print(f'Model A  MAE : {mae_a:.2f} pts')
print(f'Improvement  : {diff:+.2f} pts  ({"better" if diff > 0 else "worse"})')

Baseline MAE : 7.68 pts
Model A  MAE : 6.25 pts
Improvement  : +1.43 pts  (better)


## Putting the numbers in context

The mean of actual_peak_value is around 62 pts on a 0–100 scale. Therefore:

- Baseline MAE ≈ 7.68 pts - error ~12% of the scale
- Model A MAE ≈ 6.25 pts  - error ~10% of the scale
- Improvement: 1.43 pts - ~18% reduction in error

This looks moderate, but we need to verify it is statistically significant - with only 32 test instances, random variation is high.

In [11]:
# Permutation test: how often does a random predictor achieve MAE <= Model A?
# Low p-value => Model A predicts something real, not just noise.
perm_maes = np.array([mae(rng.permutation(actual), pred_a) for _ in range(N_PERM)])
p_value   = float(np.mean(perm_maes <= mae_a))

print(f'Permutation test  (n = {N_PERM})')
print(f'  Observed MAE (Model A)   : {mae_a:.2f}')
print(f'  Permutation MAE mean     : {perm_maes.mean():.2f}  (std {perm_maes.std():.2f})')
print(f'  p-value                  : {p_value:.4f}')

Permutation test  (n = 1000)
  Observed MAE (Model A)   : 6.25
  Permutation MAE mean     : 11.32  (std 1.24)
  p-value                  : 0.0000


## Interpreting the permutation test

p ≈ 0.000 means that none of the 1 000 random shuffles of the target values achieves an MAE as low as Model A. In other words: the model's predictions are so well aligned with the actual values that it is practically impossible for this to be coincidental.

Important distinction: the permutation test checks whether the model predicts something real (vs. random noise). It does not check whether it outperforms the baseline - that is what the bootstrap CI is for.

In [13]:
# Bootstrap CI on the MAE improvement (Model A vs Baseline)
n = len(actual)
boot_diffs = []
for _ in range(N_BOOT):
    idx = rng.integers(0, n, size=n)
    boot_diffs.append(mae(actual[idx], pred_base[idx]) - mae(actual[idx], pred_a[idx]))
boot_diffs = np.array(boot_diffs)
ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])

print(f'Bootstrap CI on improvement  (n = {N_BOOT} resamples)')
print(f'Mean improvement: {boot_diffs.mean():.2f} pts')
print(f'95% CI: [{ci_lo:.2f}, {ci_hi:.2f}] pts')

Bootstrap CI on improvement  (n = 500 resamples)
Mean improvement: 1.39 pts
95% CI: [-0.70, 3.51] pts
